In [ ]:
import re
from collections import Counter, defaultdict
from sklearn.feature_extraction.text import CountVectorizer
import matplotlib.pyplot as plt
import plotly_express as px

import numpy as np
import pandas as pd

import nltk
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('stopwords', quiet=True)
from nltk.stem.porter import PorterStemmer
from nltk.corpus import stopwords as nltk_stopwords
from sklearn.preprocessing import normalize
from sklearn.decomposition import PCA, TruncatedSVD as SVD

In [ ]:
tfidf = pd.read_csv('tfidf.csv')
tfidfL2 = pd.read_csv('tfidfL2.csv')
lib = pd.read_csv('LIB.csv')
corpus = pd.read_csv('CORPUS.csv')
vocab = pd.read_csv('VOCAB.csv')

In [ ]:
OHCO = ['album_id', 'song_id', 'token_id']
bags = dict(
    ALBUM = OHCO[:2],
    SONG = OHCO[:1]
)
token = corpus.set_index(OHCO).dropna()

In [ ]:
bag = 'ALBUM'

bags[bag]+['term_str']
bow = token.groupby(bags[bag]+['term_str']).term_str.count().to_frame('n') 
bow.head()

In [ ]:
salex_file = 'salex_nrc.csv'
SALEX = pd.read_csv(salex_file).set_index('term_str')
SALEX.columns = [col.replace('nrc_','') for col in SALEX.columns]
SALEX

In [ ]:
VOCAB_SENT = vocab.set_index("term_str").join(SALEX, how="inner")
VOCAB_SENT

In [ ]:
sent_cols = SALEX.columns.tolist()
BOW_SENT = (
    bow.reset_index()
    .merge(SALEX.reset_index()[["term_str"] + sent_cols], on="term_str", how="inner")
    .set_index(["album_id", "song_id", "term_str"])
)
BOW_SENT

In [ ]:
SENT = (
    BOW_SENT
    .assign(weighted=lambda d: d["n"] * d["sentiment"])
    .groupby(bags[bag])
    [["weighted", "n"]]
    .sum()
    .rename(columns={"n": "n_sent_words"})
)
SENT["mean_sentiment"] = SENT["weighted"] / SENT["n_sent_words"]
SENT

In [ ]:
sent_meta = (
    SENT.reset_index()
    .merge(
        lib[["song_id", "album", "release_date"]].drop_duplicates("song_id"),
        on="song_id",
        how="left",
    )
)
sent_meta["release_date"] = pd.to_datetime(sent_meta["release_date"])

album_order = (
    sent_meta.groupby("album")["release_date"]
    .min()
    .sort_values()
    .index.tolist()
)

album_means = sent_meta.groupby("album")["mean_sentiment"].mean().reindex(album_order)

short_names = {
    "A Star Is Born Soundtrack (Without Dialogue)": "A Star Is Born",
    "The Fame Monster (Deluxe Edition)": "The Fame Monster",
    "Joanne (Deluxe)": "Joanne",
}
labels = [short_names.get(a, a) for a in album_order]

fig, ax = plt.subplots(figsize=(13, 6))
colors = plt.cm.tab10.colors

bars = ax.bar(
    range(len(album_order)),
    album_means.values,
    color=[colors[i % len(colors)] for i in range(len(album_order))],
    edgecolor="k",
    linewidth=0.8,
)

ax.axhline(0, color="k", lw=0.9, linestyle="--", alpha=0.4)
ax.set_xticks(range(len(album_order)))
ax.set_xticklabels(labels, rotation=28, ha="right", fontsize=9)
ax.set_ylabel("Average Sentiment Score (NRC)")
ax.set_title("Average Song Sentiment Across Lady Gaga's Albums (chronological)")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


### W2V

In [ ]:
from gensim.models import Word2Vec

sentences = corpus.groupby(['album_id', 'song_id'])['term_str'].apply(list).tolist()

w2v_model = Word2Vec(sentences, vector_size=100, window=5, min_count=2, workers=4, seed=42, epochs=20)

In [ ]:
w2v_terms = [t for t in vocab['term_str'] if t in w2v_model.wv]

w2v_vectors = pd.DataFrame(
    [w2v_model.wv[t] for t in w2v_terms],
    index=w2v_terms,
    columns=[f"w2v_{i}" for i in range(w2v_model.wv.vector_size)],
)
w2v_vectors.index.name = 'term_str'

VOCAB_W2V = (
    vocab.set_index('term_str')[['n', 'df', 'dfidf', 'max_pos_group', 'stop']]
    .join(w2v_vectors, how='inner')
    .sort_values('n', ascending=False)
)

print(f"{len(VOCAB_W2V)} terms with Word2Vec embeddings")
VOCAB_W2V.iloc[:, :12]

### WORD2VEC: t-SNE PLOT

In [ ]:
from sklearn.manifold import TSNE

plot_terms = VOCAB_W2V[(~VOCAB_W2V['stop']) & (VOCAB_W2V['n'] >= 5)]

embedding_matrix = plot_terms[[c for c in plot_terms.columns if c.startswith('w2v_')]].values

tsne = TSNE(n_components=2, perplexity=30, random_state=42)
tsne_coords = tsne.fit_transform(embedding_matrix)

TSNE_DF = pd.DataFrame(tsne_coords, columns=['x', 'y'], index=plot_terms.index)
TSNE_DF['n'] = plot_terms['n']
TSNE_DF['pos'] = plot_terms['max_pos_group']

px.scatter(
    TSNE_DF.reset_index(), 'x', 'y',
    text='term_str',
    hover_name='term_str',
    color='pos',
    height=1000,
    width=1200,
).update_traces(
    mode='markers+text',
    textfont=dict(color='black', size=14, family='Arial'),
    textposition='top center',
)